You are right. For Jupyter Markdown, use dollar math delimiters only: inline math as $...$, display math as $$...$$. Avoid \(...\) and \[...\] because they often fail depending on the Jupyter frontend.

Paste this directly into a Markdown cell:

Current PIGNN-Attn-LS

Here, $r$ denotes the inner attention layer, while $k$ denotes the PIGNN solver iteration. GridFM has only $r$, not $k$.

T1 — Transformer-Aware Branch Model

For branch $e=(f,t)$, define the complex tap as

$$
a_e = \tau_e e^{j\varphi_e},
$$

where $\tau_e$ is the tap magnitude and $\varphi_e$ is the phase shift.

Using direction-dependent series and shunt quantities,

$$
\begin{aligned}
Y_{ff,e} &= \frac{y_{f,e}+y^{\mathrm{sh}}{f,e}/2}{|a_e|^2},
&
Y{ft,e} &= -\frac{y_{ft,e}}{a_e^*},
\
Y_{tf,e} &= -\frac{y_{tf,e}}{a_e},
&
Y_{tt,e} &= y_{t,e}+y^{\mathrm{sh}}_{t,e}/2 .
\end{aligned}
$$

These terms, together with bus shunts, form $Y_{\mathrm{bus}}$.

For directed attention edges, use

$$
\ell_{f\to t}

\left[
\Re(Y_{ft}),
\Im(Y_{ft}),
\Re(y_f^{\mathrm{sh}}),
\Im(y_f^{\mathrm{sh}}),
\Re(y_t^{\mathrm{sh}}),
\Im(y_t^{\mathrm{sh}}),
\tau,
\varphi,
\mathbb{1}_{\mathrm{trafo}}
\right].
$$

The reverse direction $t\to f$ uses $Y_{tf}$, swaps the shunts, and replaces $\varphi$ by $-\varphi$.

⸻

T2 — Physics State at Solver Step $k$

Let

$$
\underline{V}_i^{(k)}

v_i^{(k)} e^{j\theta_i^{(k)}} ,
$$

and

$$
\underline{S}^{(k)}

\underline{V}^{(k)}
\odot
\left(
Y_{\mathrm{bus}}\underline{V}^{(k)}
\right)^* .
$$

Then,

$$
\Delta P_i^{(k)}

P_i^{\mathrm{set}}

\Re\left(\underline{S}_i^{(k)}\right),
$$

and

$$
\Delta Q_i^{(k)}

Q_i^{\mathrm{set}}

\Im\left(\underline{S}_i^{(k)}\right).
$$

Operational masks impose

$$
\Delta P_i^{(k)} = 0
\qquad
\text{for } i\in\mathcal{V}_{\mathrm{slack}},
$$

and

$$
\Delta Q_i^{(k)} = 0
\qquad
\text{for } i\in\mathcal{V}{\mathrm{slack}}\cup\mathcal{V}{\mathrm{PV}}.
$$

⸻

T3 — Node Embedding

The current node input is

$$
b_i^{(k)}

\left[
v_i^{(k)},
\theta_i^{(k)},
\Delta P_i^{(k)},
\Delta Q_i^{(k)},
\nu_i,
m_i^{(k)}
\right],
$$

where

$$
\nu_i

\log_{10}
\left(
V_{i,\mathrm{nom}}^{\mathrm{kV}} + 10^{-9}
\right)
$$

when vn_feat is enabled, and is omitted otherwise.

The embedding is

$$
x_i^{(k,0)}

W_{\mathrm{in}} b_i^{(k)} + b_{\mathrm{in}} .
$$

⸻

T4 — Sparse Edge-Conditioned Attention

For attention block $r=0,\ldots,R-1$ and head $h$,

$$
\begin{aligned}
\bar{x}_i^{(k,r)}
&=
\operatorname{LN}_1
\left(
x_i^{(k,r)}
\right),
\
q_i^{(k,r,h)}
&=
W_Q^{(r,h)}
\bar{x}_i^{(k,r)},
\
k_j^{(k,r,h)}
&=
W_K^{(r,h)}
\bar{x}_j^{(k,r)},
\
z_j^{(k,r,h)}
&=
W_V^{(r,h)}
\bar{x}_j^{(k,r)} .
\end{aligned}
$$

The directed edge-conditioned score is

$$
s_{ij}^{(k,r,h)}

\frac{
\left\langle
q_i^{(k,r,h)},
k_j^{(k,r,h)}
\right\rangle
}{
\sqrt{d_h}
}
+
f_{\mathrm{edge}}^{(r,h)}
\left(
\ell_{j\to i}
\right).
$$

The attention coefficient is

$$
\alpha_{ij}^{(k,r,h)}

\operatorname{softmax}{j\in\mathcal{N}(i)}
s{ij}^{(k,r,h)} .
$$

Each block uses attention and feed-forward residual connections:

$$
\begin{aligned}
\tilde{x}i^{(k,r)}
&=
x_i^{(k,r)}
+
W_O^{(r)}
\left(
\bigg|{h=1}^{H}
\sum_{j\in\mathcal{N}(i)}
\alpha_{ij}^{(k,r,h)}
z_j^{(k,r,h)}
\right),
\
x_i^{(k,r+1)}
&=
\tilde{x}_i^{(k,r)}
+
\operatorname{FFN}^{(r)}
\left(
\operatorname{LN}_2
\left(
\tilde{x}_i^{(k,r)}
\right)
\right).
\end{aligned}
$$

Thus, $R=\texttt{num_attn_layers}$ attention blocks are executed inside every solver iteration $k$.

⸻

T5 — Iteration-Specific Update Heads

From $x_i^{(k,R)}$,

$$
\begin{aligned}
d\theta_i^{(k)}
&=
L_{\theta,k}
\left(
x_i^{(k,R)}
\right),
\
dv_i^{(k)}
&=
L_{v,k}
\left(
x_i^{(k,R)}
\right),
\
dm_i^{(k)}
&=
\operatorname{LN}
\left[
\tanh
\left(
L_{m,k}
\left(
x_i^{(k,R)}
\right)
\right)
\right].
\end{aligned}
$$

Masks and limits enforce

$$
d\theta_i^{(k)} = 0
\qquad
\text{for } i\in\mathcal{V}_{\mathrm{slack}},
$$

$$
dv_i^{(k)} = 0
\qquad
\text{for } i\in\mathcal{V}{\mathrm{slack}}\cup\mathcal{V}{\mathrm{PV}},
$$

and

$$
|d\theta_i^{(k)}|
\leq
d\theta_{\max},
\qquad
|dv_i^{(k)}|
\leq
\eta_v |v_i^{(k)}| .
$$

⸻

T6 — Armijo-Controlled State Update

For candidate step size $\alpha$,

$$
v_i^{\mathrm{try}}(\alpha)

\operatorname{clip}
\left(
v_i^{(k)}+\alpha dv_i^{(k)},
0.75,
1.20
\right),
$$

and

$$
\theta_i^{\mathrm{try}}(\alpha)

\operatorname{wrap}
\left(
\theta_i^{(k)}+\alpha d\theta_i^{(k)}
\right).
$$

Define the masked infinity mismatch

$$
F(v,\theta)

\max
\left{
\left|
\Delta P_{\mathrm{non\text{-}slack}}
\right|{\infty},
\left|
\Delta Q{\mathrm{PQ}}
\right|_{\infty}
\right}.
$$

A candidate is accepted when

$$
F
\left(
v^{\mathrm{try}},
\theta^{\mathrm{try}}
\right)
\leq
(1-c_1\alpha)
F
\left(
v^{(k)},
\theta^{(k)}
\right).
$$

The candidate sequence is approximately

$$
\alpha
\in
{1,\rho,\rho^2,\ldots},
\qquad
\alpha\geq\alpha_{\min}.
$$

The update is

$$
\left(
v^{(k+1)},
\theta^{(k+1)},
m^{(k+1)}
\right)

\left(
v^{(k)},
\theta^{(k)},
m^{(k)}
\right)
+
\alpha_k
\left(
dv^{(k)},
d\theta^{(k)},
dm^{(k)}
\right).
$$

Mode-specific behavior:

1. fixed: accept sufficient decrease; otherwise apply the smallest candidate only if it strictly reduces $F$.
2. geometric: always apply the selected candidate, including the final candidate if none passes.
3. geometric_safe: use $\alpha_{\min}$ when all candidates fail, preserving gradient flow.
4. reject: discard the entire state update when all candidates fail.

⸻

T7 — Training Objective

The physics loss uses the mismatch evaluated before each update:

$$
\mathcal{L}_{\mathrm{phys}}

\sum_{k=0}^{K-1}
\gamma^{K-1-k}
,
\ell_{\mathrm{phys}}
\left(
\Delta P^{(k)},
\Delta Q^{(k)}
\right)
+
w_f
\ell_{\mathrm{phys}}
\left(
\Delta P^{(K)},
\Delta Q^{(K)}
\right).
$$

The supervised voltage loss is

$$
\mathcal{L}_{\mathrm{NR}}

\frac{1}{N}
\sum_i
\left[
\left(
\hat{v}_i-v_i^{\mathrm{NR}}
\right)^2
+
\operatorname{wrap}
\left(
\hat{\theta}_i-\theta_i^{\mathrm{NR}}
\right)^2
\right].
$$

The combined mode uses

$$
\boxed{
\mathcal{L}_{\mathrm{PIGNN}}

\mathcal{L}{\mathrm{phys}}
+
w{\mathrm{MSE}}
\mathcal{L}_{\mathrm{NR}}
}
$$

⸻

GridFM HGNS Used Here

GridFM is a direct heterogeneous graph surrogate, not a $K$-step solver.

G1 — Heterogeneous Graph

Define

$$
\mathcal{G}

\left(
\mathcal{V}{\mathrm{bus}},
\mathcal{V}{\mathrm{gen}},
\mathcal{E}{bb},
\mathcal{E}{gb},
\mathcal{E}_{bg}
\right).
$$

The relations are

$$
\mathrm{bus}
\xrightarrow{\mathrm{branch}}
\mathrm{bus},
\qquad
\mathrm{gen}
\to
\mathrm{bus},
\qquad
\mathrm{bus}
\to
\mathrm{gen}.
$$

The bus input contains signed-log transformed injections, starting voltage, bus type, limits, shunts, and nominal voltage:

$$
x_i^{\mathrm{bus}}

\left[
\operatorname{slog}(-P_i),
\operatorname{slog}(-Q_i),
Q_{G,i},
v_i^0,
\theta_i^0,
\mathbb{1}{\mathrm{PQ}},
\mathbb{1}{\mathrm{PV}},
\mathbb{1}{\mathrm{REF}},
v_i^{\min},
v_i^{\max},
Q_i^{\min},
Q_i^{\max},
\operatorname{slog}
\left(
G_i^{\mathrm{sh}}
\right),
\operatorname{slog}
\left(
B_i^{\mathrm{sh}}
\right),
\log{10}
V_{i,\mathrm{nom}}^{\mathrm{kV}}
\right].
$$

⸻

G2 — Transformer-Aware Branch Features

For each directed branch,

$$
e_{j\to i}

\left[
\operatorname{slog}
\left(
\Re(Y_{jj})
\right),
\operatorname{slog}
\left(
\Im(Y_{jj})
\right),
\operatorname{slog}
\left(
\Re(Y_{ji})
\right),
\operatorname{slog}
\left(
\Im(Y_{ji})
\right),
\tau,
-\pi,
\pi,
0
\right],
$$

with the appropriate $Y_{ff},Y_{ft}$ or $Y_{tt},Y_{tf}$ terms from T1.

The implementation stores ten edge channels, with unused limit and rating channels set to fixed defaults.

⸻

G3 — Heterogeneous Transformer Layers

Initial projections are

$$
h_i^{\mathrm{bus},(0)}

f_{\mathrm{bus}}
\left(
x_i^{\mathrm{bus}}
\right),
$$

$$
h_g^{\mathrm{gen},(0)}

f_{\mathrm{gen}}
\left(
x_g^{\mathrm{gen}}
\right),
$$

and

$$
\tilde{e}_{ji}

f_{\mathrm{edge}}
\left(
e_{ji}
\right).
$$

For relation $r:s\to t$, layer $\ell$, and head $h$,

$$
s_{ij,r}^{(\ell,h)}

\frac{
\left(
W_{Q,r}^{(\ell,h)}
h_i^{(\ell)}
\right)^{\top}
\left(
W_{K,r}^{(\ell,h)}
h_j^{(\ell)}
+
W_{E,r}^{(\ell,h)}
\tilde{e}_{ji}
\right)
}{
\sqrt{d_h}
}.
$$

The attention coefficient is

$$
\alpha_{ij,r}^{(\ell,h)}

\operatorname{softmax}_{j\in\mathcal{N}r(i)}
s{ij,r}^{(\ell,h)} .
$$

The message is

$$
\mu_{ij,r}^{(\ell,h)}

\alpha_{ij,r}^{(\ell,h)}
\left(
W_{V,r}^{(\ell,h)}
h_j^{(\ell)}
+
W_{E,r}^{(\ell,h)}
\tilde{e}_{ji}
\right).
$$

Messages from heterogeneous relations are summed:

$$
\tilde{h}_i^{(\ell+1)}

\sum_{r:\operatorname{dst}(r)=\operatorname{type}(i)}
\left(
\bigg|{h=1}^{H}
\sum{j\in\mathcal{N}r(i)}
\mu{ij,r}^{(\ell,h)}
\right).
$$

After normalization and activation,

$$
h_i^{(\ell+1)}

h_i^{(\ell)}
+
\operatorname{LeakyReLU}
\left(
\operatorname{LN}
\left(
\tilde{h}_i^{(\ell+1)}
\right)
\right),
$$

whenever dimensions match. Otherwise, the new representation replaces the old one.

⸻

G4 — Direct Voltage Prediction

After $L$ HGNS layers,

$$
\left[
\delta v_i,
\delta\theta_i
\right]

\operatorname{MLP}_{\mathrm{out}}
\left(
h_i^{(L)}
\right).
$$

The final prediction is

$$
\boxed{
\hat{v}_i

\operatorname{clip}
\left(
v_i^0+\delta v_i,
v_{\min},
v_{\max}
\right),
\qquad
\hat{\theta}_i

\operatorname{wrap}
\left(
\theta_i^0+\delta\theta_i
\right)
}
$$

There is no intermediate voltage state $V^{(k)}$, mismatch recomputation, Armijo search, or operational output mask.

⸻

G5 — GridFM Objective

The supervised term is

$$
\mathcal{L}_{\mathrm{MSE}}

\frac{1}{N}
\sum_i
\left[
\left(
\hat{v}_i-v_i^{\mathrm{NR}}
\right)^2
+
\operatorname{wrap}
\left(
\hat{\theta}_i-\theta_i^{\mathrm{NR}}
\right)^2
\right].
$$

Physics is evaluated once at the final prediction:

$$
\hat{\underline{V}}_i

\hat{v}_i e^{j\hat{\theta}_i},
$$

and

$$
\hat{\underline{S}}

\hat{\underline{V}}
\odot
\left(
Y_{\mathrm{bus}}
\hat{\underline{V}}
\right)^* .
$$

The residual vector is

$$
r

\left[
\left(
P^{\mathrm{set}}

\Re(\hat{S})
\right)_{\mathrm{non\text{-}slack}},
\left(
Q^{\mathrm{set}}

\Im(\hat{S})
\right)_{\mathrm{PQ}}
\right].
$$

For the experiments used here,

$$
\ell_{\mathrm{phys}}(r)

\frac{1}{|r|}
\sum_q
\log
\cosh
\left(
\operatorname{clip}
\left(
r_q,
-30,
30
\right)
\right).
$$

The total GridFM loss is

$$
\boxed{
\mathcal{L}_{\mathrm{GridFM}}

w_{\mathrm{MSE}}
\mathcal{L}{\mathrm{MSE}}
+
w{\mathrm{phys}}
\ell_{\mathrm{phys}}(r)
}
$$

with

$$
w_{\mathrm{MSE}}=1,
\qquad
w_{\mathrm{phys}}=0.01 .
$$

⸻

Central Distinction

The main difference is

$$
\boxed{
\text{PIGNN-Attn-LS: }
K
\text{ learned solver updates, each containing }
R
\text{ attention blocks}
}
$$

versus

$$
\boxed{
\text{GridFM HGNS: }
L
\text{ heterogeneous Transformer layers followed by one direct voltage update}
}
$$

In [ ]:
ㅇ